In [48]:
import xarray as xr
import os
import pandas as pd

Juntando todos os arquivo .nc do conjunto de dados era5 e era5 land


In [49]:

pasta_era5_land = "/home/lacrio2/Documentos/LACRIO/PROJETO/LACRIO/Analises/ERA5_dados/dados_era5_land"
pasta_era5 = "/home/lacrio2/Documentos/LACRIO/PROJETO/LACRIO/Analises/ERA5_dados/dados_era5"
# Caminho para a pasta onde estão os arquivos .nc
arquivos_nc_land = sorted([os.path.join(pasta_era5_land, f) for f in os.listdir(pasta_era5_land) if f.endswith(".nc")])

# Abrir e concatenar ao longo da dimensão "time" (ou outra, se necessário)
ds_era5_land = xr.open_mfdataset(arquivos_nc_land, combine='by_coords')

# Caminho para a pasta onde estão os arquivos .nc
arquivos_nc = sorted([os.path.join(pasta_era5, f) for f in os.listdir(pasta_era5) if f.endswith(".nc")])

# Abrir e concatenar ao longo da dimensão "time" (ou outra, se necessário)
ds_era5 = xr.open_mfdataset(arquivos_nc, combine='by_coords')


Criando um arquivo csv para a localização da estação Cuchillacocha para o arquivo era5 land

In [50]:


# Coordenadas da estação
lat_estacao = -9.41
lon_estacao = -77.35

# Abrir o arquivo NetCDF
ds = ds_era5_land

# Verifique os nomes corretos das dimensões
print(ds.dims)
print(ds.coords)

# Substituir nomes de coordenadas se necessário
if 'lat' in ds.coords and 'lon' in ds.coords:
    ds = ds.rename({'lat': 'latitude', 'lon': 'longitude'})

# Encontrar os índices mais próximos da coordenada desejada
lat_idx = abs(ds['latitude'] - lat_estacao).argmin()
lon_idx = abs(ds['longitude'] - lon_estacao).argmin()

# Selecionar os dados no ponto mais próximo
ponto = ds.isel(latitude=lat_idx, longitude=lon_idx)

# Usar 'valid_time' como eixo temporal
tempo = ds['valid_time'].values

# Extrair dados para todas as variáveis dependentes de 'valid_time'
dados = {}
for var in ds.data_vars:
    dims = ds[var].dims
    if 'valid_time' in dims:
        dados[var] = ponto[var].values
    else:
        # Repete o valor único ao longo do tempo
        dados[var] = [ponto[var].values] * len(tempo)

# Criar DataFrame
df = pd.DataFrame(dados)
df["valid_time"] = tempo
ds_era5_land_csv = df[["valid_time"] + [v for v in dados if v != "valid_time"]]



FrozenMappingWarningOnValuesAccess({'valid_time': 118344, 'latitude': 6, 'longitude': 4})
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 947kB 1940-01-01 ... 2020-12-31T1...
  * latitude    (latitude) float64 48B -8.65 -8.9 -9.15 -9.4 -9.65 -9.9
  * longitude   (longitude) float64 32B -77.9 -77.65 -77.4 -77.15
    number      int64 8B 0
    expver      (valid_time) object 947kB dask.array<chunksize=(1464,), meta=np.ndarray>


Criando um arquivo csv para a localização da estação Cuchillacocha para o arquivo era5 

In [51]:
import xarray as xr
import pandas as pd

# Coordenadas do ponto de interesse (Cuchillacocha)
lat_estacao = -9.41
lon_estacao = -77.35

# Abrir dataset
ds = ds_era5

# Renomear lat/lon se necessário
if 'lat' in ds.coords or 'lon' in ds.coords:
    ds = ds.rename({'lat': 'latitude', 'lon': 'longitude'})

# Achar índice do ponto mais próximo
lat_idx = abs(ds['latitude'] - lat_estacao).argmin()
lon_idx = abs(ds['longitude'] - lon_estacao).argmin()

# Verifica se existe a dimensão pressure_level
if "pressure_level" not in ds.dims:
    raise ValueError("O dataset não possui a dimensão 'pressure_level'.")

# Criar dicionário com DataFrames por nível de pressão
dfs_por_pressao = {}

for nivel in ds.pressure_level.values:
    ponto_nivel = ds.isel(latitude=lat_idx, longitude=lon_idx).sel(pressure_level=nivel)
    tempo = ds["valid_time"].values
    dados = {}

    for var in ds.data_vars:
        da = ponto_nivel[var]
        if "valid_time" in da.dims:
            # Reduz outras dimensões, se houver
            dims_extras = [d for d in da.dims if d != "valid_time"]
            if dims_extras:
                da = da.isel({d: 0 for d in dims_extras})
            dados[var] = da.values
        else:
            dados[var] = [da.values.item()] * len(tempo)

    df = pd.DataFrame(dados)
    df["valid_time"] = tempo
    df = df[["valid_time"] + [v for v in dados if v != "valid_time"]]

    dfs_por_pressao[int(nivel)] = df

# Empilhar todos os DataFrames com coluna 'pressure_level'
lista_df = []

for nivel, df in dfs_por_pressao.items():
    df_com_nivel = df.copy()
    df_com_nivel["pressure_level"] = nivel
    lista_df.append(df_com_nivel)

df_empilhado = pd.concat(lista_df, ignore_index=True)

# Reorganizar colunas (opcional)
colunas_ordenadas = ["valid_time", "pressure_level"] + [col for col in df_empilhado.columns if col not in ["valid_time", "pressure_level"]]
df_era5_csv = df_empilhado[colunas_ordenadas]

# ✅ df_empilhado agora contém todos os dados organizados por tempo e nível de pressão
print("✅ DataFrame final criado com sucesso!")

# Exemplo de uso:
# print(df_empilhado.head())


✅ DataFrame final criado com sucesso!


In [52]:
# Supondo que o seu DataFrame se chama df_era5_csv
# Primeiro, garantimos que 'valid_time' esteja como índice
df = df_era5_csv.set_index(['valid_time', 'pressure_level'])

# Agora reorganizamos o DataFrame para que cada variável tenha colunas separadas por pressão
df_era5_csv = df.unstack(level='pressure_level')

# Ajusta o nome das colunas para o formato desejado (ex: z_500)
df_era5_csv.columns = [f"{var}_{press}" for var, press in df_era5_csv.columns]

# Resetando o índice, se desejar
df_era5_csv = df_era5_csv.reset_index()


In [53]:
ds_era5_land_csv_sem_nan = ds_era5_land_csv.dropna()  # Remove colunas com todos os valores NaN
df_era5_csv_sem_nan = df_era5_csv.dropna()  # Remove colunas com todos os valores NaN

Fazendo a média e acumulação para cada variável era 5 land

In [54]:
import pandas as pd

# Copiar o DataFrame original
df = ds_era5_land_csv_sem_nan.copy()

# Converter coluna de tempo (se necessário)
df["valid_time"] = pd.to_datetime(df["valid_time"])
df["date"] = df["valid_time"].dt.date

# Listas de variáveis
variaveis_instantaneas = ["u10", "v10", "d2m", "t2m", "sp", "z"]
variaveis_acumuladas   = ["tp", "ssrd", "strd", "sf"]

# Média das variáveis instantâneas
df_instantaneas = df.groupby("date")[variaveis_instantaneas].mean().reset_index()

# Soma das acumuladas
df_acumuladas = df.groupby("date")[variaveis_acumuladas].sum().reset_index()

# Conversões:
# - tp → mm
df_acumuladas["tp"] = df_acumuladas["tp"] * 1000

# - t2m → °C
df_instantaneas["t2m"] = df_instantaneas["t2m"] - 273.15

# (Opcional: d2m também costuma ser em Kelvin → podemos converter para °C também se quiser!)
df_instantaneas["d2m"] = df_instantaneas["d2m"] - 273.15

# Juntar tudo
df_era_5_land_completo = pd.merge(df_instantaneas, df_acumuladas, on="date")

# Exemplo de visualização
df_era_5_land_completo


,date,u10,v10,d2m,t2m,sp,z,tp,ssrd,strd,sf
0,1940-01-01,0.328639,0.214869,-0.051605,6.205353,61904.390625,40786.863281,0.029278,3277854.50,1974766.25,0.000000e+00
1,1940-01-02,0.588264,0.229326,1.566925,4.754242,61876.953125,40786.863281,0.586233,3717622.75,4054675.25,1.192129e-07
2,1940-01-03,0.310574,-0.197998,1.243683,4.603973,61876.234375,40786.863281,0.573084,3139294.75,4116405.50,1.311237e-07
3,1940-01-04,0.193879,-0.018040,2.397186,4.376068,61869.957031,40786.863281,1.803716,2481867.75,4220082.00,2.153246e-04
4,1940-01-05,0.398864,0.188723,2.149017,3.874969,61843.035156,40786.863281,0.953280,2525109.00,4373661.00,1.110633e-04
...,...,...,...,...,...,...,...,...,...,...,...
29216,2020-12-27,0.847211,0.476750,2.063751,3.631989,61651.832031,40786.863281,1.770064,1614066.00,4483425.00,4.876889e-04
29217,2020-12-28,0.592481,-0.031748,1.099762,2.792389,61778.027344,40786.863281,1.755811,3226387.25,4426724.00,7.736497e-04
29218,2020-12-29,0.754008,0.241566,2.066620,4.761810,61838.539062,40786.863281,1.052107,3301645.50,4403458.50,7.603457e-05
29219,2020-12-30,0.636928,0.491266,1.414825,4.944672,61817.257812,40786.863281,0.267223,3391407.00,4252856.00,1.072767e-07


Fazendo a média para cada variável era 5

In [55]:

df_era5_csv_sem_nan.rename(columns={"valid_time" : "date"}, inplace=True)

df_era5_csv_sem_nan.columns

Index(['date', 'z_300', 'z_400', 'z_500', 'q_300', 'q_400', 'q_500',
       'crwc_300', 'crwc_400', 'crwc_500', 't_300', 't_400', 't_500', 'u_300',
       'u_400', 'u_500', 'v_300', 'v_400', 'v_500'],
      dtype='object')

In [60]:
import pandas as pd

# Copiar o DataFrame
df = df_era5_csv_sem_nan.copy()

# Converter coluna de tempo (se necessário)
df["date"] = pd.to_datetime(df["date"])
df["date"] = df["date"].dt.date

# Lista de variáveis instantâneas
variaveis_instantaneas = ['z_300', 'z_400', 'z_500', 'q_300', 'q_400', 'q_500',
       'crwc_300', 'crwc_400', 'crwc_500', 't_300', 't_400', 't_500', 'u_300',
       'u_400', 'u_500', 'v_300', 'v_400', 'v_500']

# Agrupar por data + nível de pressão, tirar média
df_era_5_completo = df.groupby(["date"])[variaveis_instantaneas].mean().reset_index()

# Conversão opcional:
# - t (temperatura) K → °C
# Converte todas as colunas que começam com 't_' de Kelvin para Celsius
for col in df_era_5_completo.columns:
    if col.startswith('t_'):
        df_era_5_completo[col] = df_era_5_completo[col] - 273.15

# Exemplo de visualização
df_era_5_completo




,date,z_300,z_400,z_500,q_300,q_400,q_500,crwc_300,crwc_400,crwc_500,t_300,t_400,t_500,u_300,u_400,u_500,v_300,v_400,v_500
0,1940-01-01,94812.320312,74360.429688,57534.500000,0.000446,0.001279,0.002186,0.0,0.0,0.000000e+00,-33.647003,-17.487457,-4.616638,-6.099928,0.045611,-0.382076,-3.461634,1.453575,3.116183
1,1940-01-02,94770.718750,74318.554688,57512.160156,0.000507,0.001330,0.002043,0.0,0.0,0.000000e+00,-33.238312,-17.626190,-5.169067,-2.169100,4.247507,1.607090,0.126608,2.603156,3.086038
2,1940-01-03,94692.000000,74215.101562,57470.582031,0.000554,0.001255,0.001943,0.0,0.0,0.000000e+00,-32.392746,-17.904770,-6.141968,-4.701153,2.222451,2.297445,-0.607233,0.719061,-1.014605
3,1940-01-04,94635.023438,74158.929688,57418.480469,0.000564,0.001613,0.002860,0.0,0.0,0.000000e+00,-32.790192,-18.029892,-6.508179,-5.535532,-0.474765,0.913041,-5.240398,-3.327587,-3.543441
4,1940-01-05,94583.750000,74148.601562,57409.433594,0.000398,0.000999,0.003003,0.0,0.0,0.000000e+00,-33.625809,-17.850143,-6.630798,-6.644248,-1.505604,0.273474,-9.742602,-7.800463,-2.875338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29520,2020-11-11,95142.960938,74470.609375,57589.015625,0.000219,0.001125,0.004632,0.0,0.0,0.000000e+00,-30.994675,-15.050720,-5.012878,-1.150112,-4.430001,-2.596597,5.176749,4.556537,-0.386852
29521,2020-11-12,95189.960938,74483.390625,57600.597656,0.000248,0.000876,0.004587,0.0,0.0,0.000000e+00,-30.449570,-14.848999,-5.026764,-0.324422,-6.874577,-3.690191,2.754486,4.579809,0.317708
29522,2020-11-13,95188.460938,74533.562500,57656.660156,0.000226,0.000796,0.004199,0.0,0.0,0.000000e+00,-30.981110,-15.266205,-4.893402,0.918234,-6.230453,-3.384517,4.216425,4.998417,0.574383
29523,2020-11-14,95114.539062,74444.460938,57569.199219,0.000192,0.001001,0.003882,0.0,0.0,2.080393e-09,-31.124069,-15.358704,-4.821045,2.640176,-3.696155,-2.040295,5.630388,4.267547,1.537333


Merge dos dois dataframe


In [63]:
# Garante que ambas as colunas 'date' estejam no tipo datetime
df_era_5_completo['date'] = pd.to_datetime(df_era_5_completo['date'])
df_era_5_land_completo['date'] = pd.to_datetime(df_era_5_land_completo['date'])

# Faz o merge com base na coluna 'date'
df_era5_merged = pd.merge(df_era_5_completo, df_era_5_land_completo, on='date', how='inner')

df_era5_merged


,date,z_300,z_400,z_500,q_300,q_400,q_500,crwc_300,crwc_400,crwc_500,...,u10,v10,d2m,t2m,sp,z,tp,ssrd,strd,sf
0,1940-01-01,94812.320312,74360.429688,57534.500000,0.000446,0.001279,0.002186,0.0,0.0,0.000000e+00,...,0.328639,0.214869,-0.051605,6.205353,61904.390625,40786.863281,0.029278,3277854.50,1974766.25,0.000000e+00
1,1940-01-02,94770.718750,74318.554688,57512.160156,0.000507,0.001330,0.002043,0.0,0.0,0.000000e+00,...,0.588264,0.229326,1.566925,4.754242,61876.953125,40786.863281,0.586233,3717622.75,4054675.25,1.192129e-07
2,1940-01-03,94692.000000,74215.101562,57470.582031,0.000554,0.001255,0.001943,0.0,0.0,0.000000e+00,...,0.310574,-0.197998,1.243683,4.603973,61876.234375,40786.863281,0.573084,3139294.75,4116405.50,1.311237e-07
3,1940-01-04,94635.023438,74158.929688,57418.480469,0.000564,0.001613,0.002860,0.0,0.0,0.000000e+00,...,0.193879,-0.018040,2.397186,4.376068,61869.957031,40786.863281,1.803716,2481867.75,4220082.00,2.153246e-04
4,1940-01-05,94583.750000,74148.601562,57409.433594,0.000398,0.000999,0.003003,0.0,0.0,0.000000e+00,...,0.398864,0.188723,2.149017,3.874969,61843.035156,40786.863281,0.953280,2525109.00,4373661.00,1.110633e-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29155,2020-11-11,95142.960938,74470.609375,57589.015625,0.000219,0.001125,0.004632,0.0,0.0,0.000000e+00,...,-0.157606,0.140741,4.171204,6.688080,61930.597656,40786.863281,1.657321,3443632.75,4452861.00,2.053156e-04
29156,2020-11-12,95189.960938,74483.390625,57600.597656,0.000248,0.000876,0.004587,0.0,0.0,0.000000e+00,...,0.178194,0.155500,4.180725,6.451019,61940.175781,40786.863281,1.185718,3317819.75,4505626.00,5.292427e-05
29157,2020-11-13,95188.460938,74533.562500,57656.660156,0.000226,0.000796,0.004199,0.0,0.0,0.000000e+00,...,0.017953,0.038995,4.122803,6.587738,61970.253906,40786.863281,1.288152,3039426.50,4490739.50,1.933098e-04
29158,2020-11-14,95114.539062,74444.460938,57569.199219,0.000192,0.001001,0.003882,0.0,0.0,2.080393e-09,...,0.207066,0.205525,4.106750,6.518219,61897.375000,40786.863281,1.566299,2828654.00,4501893.50,3.185589e-05
